# Geospatial Data Analysis
### Objective
Visualize sales data geographically and detect underserved regional markets.

**Dataset note:** This is a synthetic demonstration dataset created because no source dataset was provided.

In [ ]:
# Geospatial Data Analysis
# Objective: Visualize sales data geographically and detect underserved regional markets.
# Dataset note: Synthetic demonstration dataset created because no source dataset was provided.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import MarkerCluster

# If running in Google Colab, upload geospatial_sales_data.csv first.
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("geospatial_sales_data.csv")
print("Dataset shape:", df.shape)
display(df.head())

# -------------------------
# 1. Data Cleaning
# -------------------------
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()
df["PostalCode"] = df["PostalCode"].astype(str).str.strip()
df["State"] = df["State"].astype(str).str.strip()
df["City"] = df["City"].astype(str).str.strip()
df["Users"] = pd.to_numeric(df["Users"], errors="coerce")
df["Revenue"] = pd.to_numeric(df["Revenue"], errors="coerce")
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
df = df.dropna(subset=["State","City","PostalCode","Users","Revenue","Latitude","Longitude"])

# -------------------------
# 2. Aggregate by city
# -------------------------
city = (df.groupby(["State","City","PostalCode","Latitude","Longitude"], as_index=False)
          .agg(Total_Users=("Users","sum"), Total_Revenue=("Revenue","sum")))

city["Revenue_per_User"] = city["Total_Revenue"] / city["Total_Users"]

def minmax(s):
    return (s-s.min())/(s.max()-s.min()) if s.max()!=s.min() else pd.Series(0.5,index=s.index)

city["Demand_Score"] = minmax(city["Total_Users"])
city["Coverage_Gap_Score"] = 1 - minmax(city["Revenue_per_User"])
city["Potential_Score"] = 0.65*city["Demand_Score"] + 0.35*city["Coverage_Gap_Score"]

city["Qualified"] = city["Total_Users"] >= city["Total_Users"].quantile(0.40)
top3 = city[city["Qualified"]].sort_values("Potential_Score", ascending=False).head(3)

print("Top 3 high-potential underserved regions:")
display(top3[["State","City","PostalCode","Total_Users","Total_Revenue",
              "Revenue_per_User","Potential_Score"]])

# -------------------------
# 3. State-level aggregation
# -------------------------
state = (df.groupby("State", as_index=False)
           .agg(Total_Users=("Users","sum"), Total_Revenue=("Revenue","sum")))
state["Revenue_per_User"] = state["Total_Revenue"] / state["Total_Users"]
display(state.sort_values("Total_Revenue", ascending=False))

# -------------------------
# 4. Revenue chart
# -------------------------
s = state.sort_values("Total_Revenue", ascending=False).head(10)
plt.figure(figsize=(9,5))
plt.barh(s["State"][::-1], s["Total_Revenue"][::-1]/1e6)
plt.xlabel("Revenue (₹ million)")
plt.title("Top 10 States by Total Revenue")
plt.tight_layout()
plt.show()

# -------------------------
# 5. Interactive Folium map
# -------------------------
m = folium.Map(location=[22.8,79.0], zoom_start=5, tiles="OpenStreetMap")
cluster = MarkerCluster().add_to(m)

top_names = set(top3["City"])

for _, r in city.iterrows():
    popup = f"""<b>{r['City']}, {r['State']}</b><br>
    Postal Code: {r['PostalCode']}<br>
    Users: {int(r['Total_Users']):,}<br>
    Revenue: ₹{r['Total_Revenue']:,.0f}<br>
    Revenue/User: ₹{r['Revenue_per_User']:,.0f}<br>
    Potential Score: {r['Potential_Score']:.3f}"""
    folium.CircleMarker(
        [r["Latitude"], r["Longitude"]],
        radius=9 if r["City"] in top_names else 6,
        popup=folium.Popup(popup, max_width=300),
        tooltip=f"{r['City']} — {'TOP OPPORTUNITY' if r['City'] in top_names else 'Regional Market'}",
        fill=True,
        color="red" if r["City"] in top_names else "blue",
        fill_opacity=0.7
    ).add_to(cluster)

for _, r in top3.iterrows():
    folium.Marker(
        [r["Latitude"], r["Longitude"]],
        tooltip=f"Top opportunity: {r['City']}",
        icon=folium.Icon(color="red", icon="star")
    ).add_to(m)

m
# To save the interactive map:
m.save("geospatial_map.html")
print("Map saved as geospatial_map.html")
